# Hito 2 — Generación de anuncios contextuales con IA generativa

Para cada texto de `muestra_evaluacion` (50 noticias + 50 artículos de WikiHow) se
pide a un modelo generativo (Claude, vía API de Anthropic) que, en dos pasos:

1. asigne una **categoría contextual**: una de las 12 categorías originales del
   corpus (se usan estas, y no una taxonomía publicitaria propia, para poder
   evaluar la clasificación con las mismas etiquetas del Hito 1);
2. genere un **anuncio** breve adecuado a esa categoría y al contenido del texto.

Después, dos anotadores evalúan de forma independiente la categoría y el encaje del
anuncio, y se mide el acuerdo entre ambos.

**Dos modos de generación:**

- **`api`**: si existe `ANTHROPIC_API_KEY`, cada texto se envía al modelo
  (`claude-opus-5-5`) a través de la API. La clave se guarda en un archivo `.env`
  en la raíz del repositorio (plantilla en `.env.example`). Ese archivo está en
  `.gitignore`, así que la clave nunca aparece en el notebook ni se sube a GitHub.
- **`manual`**: sin API key, el notebook lee las respuestas de
  `data/hito2/respuestas_manuales.csv`. Ese archivo se generó con Claude (Opus 5.5)
  en una sesión interactiva, aplicando a cada texto exactamente el mismo
  `crear_prompt`. El formato de las respuestas es idéntico, así que la
  extracción, la validación y la evaluación no cambian.

En esta entrega se ha usado el modo **manual**, porque no se disponía de API key.

## 1. Configuración

In [1]:
import os
import re
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "hito2"

# Carga ANTHROPIC_API_KEY desde el archivo .env (no versionado)
load_dotenv(ROOT / ".env")
MODO = "api" if os.getenv("ANTHROPIC_API_KEY") else "manual"
MODELO = "claude-opus-5-5"

if MODO == "api":
    client = anthropic.Anthropic()  # lee la clave del entorno, nunca escrita aquí
else:
    manuales = pd.read_csv(DATA / "respuestas_manuales.csv")
print(f"Modo de generación: {MODO}")

Modo de generación: manual


In [2]:
muestra_evaluacion = pd.read_csv(DATA / "muestra_evaluacion.csv")
print(muestra_evaluacion.shape)
muestra_evaluacion["source_type"].value_counts()

(100, 10)


source_type
news       50
wikihow    50
Name: count, dtype: int64

## 2. Prompt

Las categorías permitidas son las 12 del corpus del Hito 1. Se pide al modelo un
formato de salida fijo (`CATEGORIA:` / `ANUNCIO:`) para poder extraer los campos
automáticamente.

In [3]:
CATEGORIAS_PERMITIDAS = {
    "alimentation": "alimentación, gastronomía, recetas y bebidas",
    "astronomy": "astronomía, espacio y ciencias del universo",
    "economy": "economía, finanzas, empresas y empleo",
    "fashion": "moda, belleza y estilo",
    "medicine": "medicina, salud y bienestar",
    "military": "defensa, ejército y conflictos armados",
    "motor": "motor, automóviles y vehículos",
    "play": "ocio, juegos, videojuegos y entretenimiento",
    "politics": "política, gobierno e instituciones",
    "religion": "religión y espiritualidad",
    "sport": "deporte y competiciones deportivas",
    "tech": "tecnología, informática, internet y dispositivos",
}


def crear_prompt(texto):
    lista = "\n".join(f"- {cat}: {desc}" for cat, desc in CATEGORIAS_PERMITIDAS.items())
    return f"""Eres un sistema de publicidad contextual. Vas a leer el texto de una página web.

Tareas:
1. Clasifica el texto en UNA sola categoría de la siguiente lista. Escribe exactamente el identificador (la palabra antes de los dos puntos), aunque el encaje no sea perfecto elige la más cercana.
{lista}
2. Redacta un anuncio breve en español (una o dos frases, máximo 30 palabras) de un producto o servicio adecuado a la categoría asignada y coherente con el contenido concreto del texto.

Responde exactamente con este formato y nada más:
CATEGORIA: <identificador>
ANUNCIO: <texto del anuncio>

Texto:
<<<
{texto}
>>>"""


print(crear_prompt("Texto de ejemplo."))

Eres un sistema de publicidad contextual. Vas a leer el texto de una página web.

Tareas:
1. Clasifica el texto en UNA sola categoría de la siguiente lista. Escribe exactamente el identificador (la palabra antes de los dos puntos), aunque el encaje no sea perfecto elige la más cercana.
- alimentation: alimentación, gastronomía, recetas y bebidas
- astronomy: astronomía, espacio y ciencias del universo
- economy: economía, finanzas, empresas y empleo
- fashion: moda, belleza y estilo
- medicine: medicina, salud y bienestar
- military: defensa, ejército y conflictos armados
- motor: motor, automóviles y vehículos
- play: ocio, juegos, videojuegos y entretenimiento
- politics: política, gobierno e instituciones
- religion: religión y espiritualidad
- sport: deporte y competiciones deportivas
- tech: tecnología, informática, internet y dispositivos
2. Redacta un anuncio breve en español (una o dos frases, máximo 30 palabras) de un producto o servicio adecuado a la categoría asignada y cohe

## 3. Llamada al modelo

`generar_respuesta(texto)` construye el prompt con `crear_prompt`, llama al modelo y
devuelve el texto de la respuesta.

- En modo `api` se usa esfuerzo `low`, porque la tarea es corta. Con
  `fallbacks="default"`, si el modelo rechaza una petición por sus filtros de
  seguridad, la API la reintenta automáticamente con otro modelo.
- En modo `manual` se devuelve la respuesta ya generada para ese texto.

In [4]:
def generar_respuesta(texto):
    if MODO == "manual":
        fila = manuales.loc[manuales["text"] == texto, "respuesta_modelo"]
        return fila.iloc[0] if len(fila) else ""
    respuesta = client.beta.messages.create(
        model=MODELO,
        max_tokens=2048,
        output_config={"effort": "low"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        messages=[{"role": "user", "content": crear_prompt(texto)}],
    )
    if respuesta.stop_reason == "refusal":
        return ""
    return "".join(b.text for b in respuesta.content if b.type == "text").strip()


def extraer_campos(respuesta):
    """Extrae (categoria_generada, anuncio_generado) del texto devuelto por el modelo."""
    cat = re.search(r"CATEGOR[IÍ]A\s*:\s*(.+)", respuesta, flags=re.IGNORECASE)
    anuncio = re.search(r"ANUNCIO\s*:\s*(.+)", respuesta, flags=re.IGNORECASE | re.DOTALL)
    categoria = cat.group(1).strip().strip("*`\"'.[]<> ").lower() if cat else None
    anuncio = " ".join(anuncio.group(1).split()).strip("*\" ") if anuncio else None
    return categoria, anuncio

### 3.1 Prueba con 1 texto

In [5]:
texto = muestra_evaluacion.loc[0, "text"]
respuesta = generar_respuesta(texto)
print(respuesta)
print(extraer_campos(respuesta))

CATEGORIA: sport
ANUNCIO: Vive la eliminatoria España-Serbia rumbo al Mundial 2025 con la camiseta oficial de los Hispanos y entradas para el partido de ida. ¡Reserva ya!
('sport', 'Vive la eliminatoria España-Serbia rumbo al Mundial 2025 con la camiseta oficial de los Hispanos y entradas para el partido de ida. ¡Reserva ya!')


### 3.2 Prueba con 10 textos

In [6]:
prueba_10 = muestra_evaluacion.head(10).copy()
prueba_10["respuesta_modelo"] = [generar_respuesta(t) for t in tqdm(prueba_10["text"])]
prueba_10[["categoria_generada", "anuncio_generado"]] = prueba_10["respuesta_modelo"].apply(
    lambda r: pd.Series(extraer_campos(r))
)
prueba_10[["id", "source_type", "categoria_generada", "anuncio_generado"]]

  0%|          | 0/10 [00:00<?, ?it/s]

,id,source_type,categoria_generada,anuncio_generado
0,1,news,sport,Vive la eliminatoria España-Serbia rumbo al Mu...
1,2,news,economy,Auditoría y cumplimiento normativo contra el b...
2,3,news,play,¿Te dejó sin aliento 'Nadie te salvará'? Suscr...
3,4,news,play,Amplía tu Xbox Series S con un disco SSD de ex...
4,5,news,military,Congreso internacional de sistemas de defensa ...
5,6,news,astronomy,Descubre los secretos de los agujeros negros c...
6,7,news,fashion,Completa tu look de novia low cost con nuestro...
7,8,news,politics,Sigue en directo la noche electoral gallega co...
8,9,news,motor,Devuelve el brillo de concesionario a tu coche...
9,10,news,sport,Anima al Betis en la eliminatoria de Conferenc...


## 4. Generación para los 100 textos

Las respuestas ya obtenidas (incluidas las 10 de la prueba) se reutilizan para no
repetir llamadas. Si una respuesta no contiene una categoría válida, se vuelve a
pedir hasta 2 veces más.

In [7]:
respuestas = dict(zip(prueba_10["id"], prueba_10["respuesta_modelo"]))
SALIDA = DATA / "muestra_generada.csv"
if SALIDA.exists():  # reanudar si ya se generó antes
    previa = pd.read_csv(SALIDA)
    respuestas |= dict(zip(previa["id"], previa["respuesta_modelo"].fillna("")))

for fila in tqdm(muestra_evaluacion.itertuples(), total=len(muestra_evaluacion)):
    for _ in range(3):
        if extraer_campos(respuestas.get(fila.id, ""))[0] in CATEGORIAS_PERMITIDAS:
            break
        respuestas[fila.id] = generar_respuesta(fila.text)

muestra_generada = muestra_evaluacion[["id", "text", "source_type"]].copy()
muestra_generada["respuesta_modelo"] = muestra_generada["id"].map(respuestas)
muestra_generada[["categoria_generada", "anuncio_generado"]] = muestra_generada["respuesta_modelo"].apply(
    lambda r: pd.Series(extraer_campos(r))
)
muestra_generada.to_csv(SALIDA, index=False)
muestra_generada[["id", "source_type", "categoria_generada", "anuncio_generado"]].head()

  0%|          | 0/100 [00:00<?, ?it/s]

,id,source_type,categoria_generada,anuncio_generado
0,1,news,sport,Vive la eliminatoria España-Serbia rumbo al Mu...
1,2,news,economy,Auditoría y cumplimiento normativo contra el b...
2,3,news,play,¿Te dejó sin aliento 'Nadie te salvará'? Suscr...
3,4,news,play,Amplía tu Xbox Series S con un disco SSD de ex...
4,5,news,military,Congreso internacional de sistemas de defensa ...


## 5. Validación de categorías

In [8]:
muestra_generada["categoria_valida"] = muestra_generada["categoria_generada"].isin(CATEGORIAS_PERMITIDAS)
invalidas = muestra_generada.loc[~muestra_generada["categoria_valida"]]
sin_anuncio = muestra_generada["anuncio_generado"].isna().sum()

print(f"Categorías válidas: {muestra_generada['categoria_valida'].sum()}/{len(muestra_generada)}")
print(f"Respuestas sin anuncio: {sin_anuncio}")
assert invalidas.empty, f"Categorías fuera de la lista:\n{invalidas[['id', 'categoria_generada']]}"

Categorías válidas: 100/100


Respuestas sin anuncio: 0


In [9]:
pd.crosstab(muestra_generada["categoria_generada"], muestra_generada["source_type"], margins=True)

source_type,news,wikihow,All
categoria_generada,,,
alimentation,4,5,9
astronomy,3,0,3
economy,1,6,7
fashion,5,8,13
medicine,4,12,16
military,2,1,3
motor,2,1,3
play,5,9,14
politics,5,0,5


## 6. Archivos para la evaluación humana

Se crean dos archivos idénticos e independientes, uno por anotador. Cada persona
rellena **solo el suyo**, sin ver el del otro:

| Columna | Valores |
|---|---|
| `categoria_correcta` | 1 = sí, 0 = no |
| `encaje_anuncio` | 2 = encaja claramente, 1 = encaje parcial, 0 = no encaja |
| `comentario` | opcional |

Los archivos no se sobrescriben si ya existen, para no perder anotaciones.

In [10]:
plantilla = muestra_generada[["id", "source_type", "text", "categoria_generada", "anuncio_generado"]].copy()
plantilla["categoria_correcta"] = pd.NA
plantilla["encaje_anuncio"] = pd.NA
plantilla["comentario"] = ""

for nombre in ["anotador_1.csv", "anotador_2.csv"]:
    ruta = DATA / nombre
    if ruta.exists():
        print(f"{nombre} ya existe: no se sobrescribe")
    else:
        plantilla.to_csv(ruta, index=False, encoding="utf-8-sig")  # utf-8-sig: tildes correctas en Excel
        print(f"Creado {ruta.relative_to(ROOT)}")

anotador_1.csv ya existe: no se sobrescribe
anotador_2.csv ya existe: no se sobrescribe


## 7. Evaluación humana

Con las dos anotaciones completas se calcula, en conjunto y por fuente
(`news` / `wikihow`):

- **Porcentaje de acuerdo** y **Cohen's Kappa** para `categoria_correcta` y
  `encaje_anuncio`. Para el encaje, al ser una escala ordinal, se añade también la
  Kappa ponderada cuadrática (un desacuerdo 0 vs 2 penaliza más que 1 vs 2).
- **Categorías correctas según ambos**: % de textos que ambos anotadores marcan
  como correctos.
- **Encaje medio**: media de las puntuaciones de los dos anotadores (escala 0–2).

**En esta entrega solo se completó una anotación (anotador 2).** La segunda no
llegó a hacerse de forma independiente, así que no se calculan acuerdo ni Kappa,
y los resultados corresponden a un único anotador. Si se completa
`anotador_1.csv`, basta con volver a ejecutar el notebook para obtener también
las métricas de acuerdo.

In [11]:
def cargar_anotaciones(nombre):
    """Carga un archivo de anotador y devuelve (df, problemas).

    Acepta separador , o ; y codificación utf-8 o latin-1, por si se guardó desde Excel.
    """
    for encoding in ["utf-8-sig", "latin-1"]:
        try:
            df = pd.read_csv(DATA / nombre, sep=None, engine="python", encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    df = df[["id", "source_type", "categoria_correcta", "encaje_anuncio", "comentario"]].copy()
    problemas = []
    for col, validos in [("categoria_correcta", {0, 1}), ("encaje_anuncio", {0, 1, 2})]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        faltan = df[col].isna().sum()
        fuera = (~df[col].dropna().isin(validos)).sum()
        if faltan:
            problemas.append(f"{faltan} filas sin rellenar en {col}")
        if fuera:
            problemas.append(f"{fuera} valores no válidos en {col}")
    if not problemas:
        df[["categoria_correcta", "encaje_anuncio"]] = df[["categoria_correcta", "encaje_anuncio"]].astype(int)
    return df, problemas


anotadores = {}
for n in [1, 2]:
    df, problemas = cargar_anotaciones(f"anotador_{n}.csv")
    if problemas:
        print(f"anotador_{n}.csv: incompleto ({'; '.join(problemas)})")
    else:
        anotadores[n] = df
        print(f"anotador_{n}.csv: completo")

DOS_ANOTADORES = len(anotadores) == 2
HAY_ANOTACIONES = len(anotadores) > 0

anotador_1.csv: incompleto (100 filas sin rellenar en categoria_correcta; 100 filas sin rellenar en encaje_anuncio)
anotador_2.csv: completo


Con **dos** anotaciones completas se calculan el acuerdo y la Kappa, y una
categoría cuenta como correcta solo si lo es para ambos anotadores. Con **una
sola** anotación no se puede medir el acuerdo entre anotadores, así que se
calculan únicamente el porcentaje de categorías correctas, el encaje medio y la
distribución del encaje.

In [12]:
def kappa(x, y, weights=None):
    # Si ambos anotadores usan un único valor idéntico, Kappa no está definida
    if len(set(x) | set(y)) < 2:
        return float("nan")
    return cohen_kappa_score(x, y, weights=weights)


def metricas_acuerdo(df):
    c1, c2 = df["categoria_correcta_1"], df["categoria_correcta_2"]
    e1, e2 = df["encaje_anuncio_1"], df["encaje_anuncio_2"]
    return pd.Series({
        "n_textos": len(df),
        "acuerdo_categoria_pct": (c1 == c2).mean() * 100,
        "kappa_categoria": kappa(c1, c2),
        "acuerdo_encaje_pct": (e1 == e2).mean() * 100,
        "kappa_encaje": kappa(e1, e2),
        "kappa_encaje_ponderada": kappa(e1, e2, weights="quadratic"),
        "categoria_correcta_ambos_pct": ((c1 == 1) & (c2 == 1)).mean() * 100,
        "categoria_correcta_anotador_1_pct": c1.mean() * 100,
        "categoria_correcta_anotador_2_pct": c2.mean() * 100,
        "encaje_medio": ((e1 + e2) / 2).mean(),
        "encaje_medio_anotador_1": e1.mean(),
        "encaje_medio_anotador_2": e2.mean(),
    })


def metricas_un_anotador(df):
    c, e = df["categoria_correcta"], df["encaje_anuncio"]
    return pd.Series({
        "n_textos": len(df),
        "categoria_correcta_pct": c.mean() * 100,
        "encaje_medio": e.mean(),
        "encaje_2_pct": (e == 2).mean() * 100,
        "encaje_1_pct": (e == 1).mean() * 100,
        "encaje_0_pct": (e == 0).mean() * 100,
    })


if DOS_ANOTADORES:
    anotaciones = anotadores[1].merge(anotadores[2].drop(columns="source_type"), on="id",
                                      suffixes=("_1", "_2"), validate="one_to_one")
    anotaciones["categoria_ok"] = anotaciones["categoria_correcta_1"] & anotaciones["categoria_correcta_2"]
    anotaciones["encaje_medio"] = (anotaciones["encaje_anuncio_1"] + anotaciones["encaje_anuncio_2"]) / 2
    calcular = metricas_acuerdo
    TIPO_EVALUACION = "dos anotadores"
elif HAY_ANOTACIONES:
    n, anotaciones = next(iter(anotadores.items()))
    anotaciones = anotaciones.copy()
    anotaciones["categoria_ok"] = anotaciones["categoria_correcta"]
    anotaciones["encaje_medio"] = anotaciones["encaje_anuncio"]
    calcular = metricas_un_anotador
    TIPO_EVALUACION = f"un anotador (anotador {n})"
    print(f"Solo hay una anotación completa (anotador {n}): no se calculan acuerdo ni Kappa.")

if HAY_ANOTACIONES:
    assert len(anotaciones) == len(muestra_generada), "Los archivos de anotación no contienen los 100 textos"
    resultados = pd.concat(
        {"total": calcular(anotaciones), **{f: calcular(g) for f, g in anotaciones.groupby("source_type")}},
        axis=1,
    ).T
    resultados.index.name = "fuente"
    resultados.insert(0, "tipo_evaluacion", TIPO_EVALUACION)
    display(resultados.round(3))
else:
    print("Anotación pendiente: las métricas se calcularán cuando haya al menos un archivo completo.")

Solo hay una anotación completa (anotador 2): no se calculan acuerdo ni Kappa.


,tipo_evaluacion,n_textos,categoria_correcta_pct,encaje_medio,encaje_2_pct,encaje_1_pct,encaje_0_pct
fuente,,,,,,,
total,un anotador (anotador 2),100.0,88.0,1.97,97.0,3.0,0.0
news,un anotador (anotador 2),50.0,100.0,2.00,100.0,0.0,0.0
wikihow,un anotador (anotador 2),50.0,76.0,1.94,94.0,6.0,0.0


### 7.1 Comparación `news` vs `wikihow`

In [13]:
if HAY_ANOTACIONES:
    comparacion = resultados.drop(columns="tipo_evaluacion").loc[["news", "wikihow"]].T
    comparacion["diferencia_news_menos_wikihow"] = comparacion["news"] - comparacion["wikihow"]
    display(comparacion.round(3))

fuente,news,wikihow,diferencia_news_menos_wikihow
n_textos,50.0,50.00,0.00
categoria_correcta_pct,100.0,76.00,24.00
encaje_medio,2.0,1.94,0.06
encaje_2_pct,100.0,94.00,6.00
encaje_1_pct,0.0,6.00,-6.00
encaje_0_pct,0.0,0.00,0.00


In [14]:
# Categorías correctas y encaje medio por categoría generada
if HAY_ANOTACIONES:
    anotaciones_cat = anotaciones.merge(muestra_generada[["id", "categoria_generada", "anuncio_generado", "text"]], on="id")
    por_categoria = anotaciones_cat.groupby("categoria_generada").agg(
        n_textos=("id", "size"),
        n_wikihow=("source_type", lambda s: (s == "wikihow").sum()),
        categoria_correcta_pct=("categoria_ok", lambda s: s.mean() * 100),
        encaje_medio=("encaje_medio", "mean"),
    ).sort_values("categoria_correcta_pct")
    display(por_categoria.round(2))

,n_textos,n_wikihow,categoria_correcta_pct,encaje_medio
categoria_generada,,,,
motor,3,1,66.67,2.00
play,14,9,71.43,1.93
medicine,16,12,81.25,2.00
tech,14,7,85.71,1.93
alimentation,9,5,88.89,2.00
fashion,13,8,92.31,1.92
economy,7,6,100.00,2.00
astronomy,3,0,100.00,2.00
military,3,1,100.00,2.00


In [15]:
# Casos con categoría incorrecta
if HAY_ANOTACIONES:
    col_comentario = "comentario" if "comentario" in anotaciones_cat else "comentario_2"
    incorrectas = anotaciones_cat.loc[anotaciones_cat["categoria_ok"] == 0,
                                      ["id", "source_type", "categoria_generada", col_comentario, "text"]].copy()
    incorrectas["text"] = incorrectas["text"].str[:80] + "…"
    with pd.option_context("display.max_colwidth", 90):
        display(incorrectas)

,id,source_type,categoria_generada,comentario,text
50,51,wikihow,medicine,Tema familiar/relacional; 'medicine' resulta forzada.,¿Cómo llevarte Bien con tus nuevos hermanastros?. Los conflictos con tus nuevos …
53,54,wikihow,medicine,"Consejos sociales y de autoestima, no un tema médico en sentido estricto.",¿Cómo hacer Que la gente te ame?. y deja que los demás conozcan quién eres realm…
59,60,wikihow,tech,Tema ambiental y de jardinería; 'tech' no describe bien el contenido.,¿Cómo prevenir La erosión del suelo?. El viento y el agua (los dos principales c…
74,75,wikihow,fashion,El texto trata principalmente de relaciones y confianza; 'fashion' solo cubre una parte.,¿Cómo ser Interesante delante de la persona que te gusta (para chicas)?. Las per…
75,76,wikihow,play,Mantenimiento de piscina; 'play' funciona aquí como categoría comodín.,¿Cómo convertir Una piscina de agua salada a cloro?. Continúa utilizando el kit …
77,78,wikihow,medicine,Autocuidado y estilo de vida; 'medicine' resulta demasiado clínica.,"¿Cómo ser Flojo?. Dependiendo de tus antecedentes y de tus creencias, la idea de…"
81,82,wikihow,play,Carpintería y bricolaje; 'play' es una asignación genérica.,¿Cómo conseguir Madera preciosa por poco dinero?. Las empresas de carpintería co…
83,84,wikihow,alimentation,Adaptación de un gato al hogar; 'alimentation' no representa el tema principal.,¿Cómo alentar A tu nuevo gato a salir de su escondite?. Al gato le costará mucho…
93,94,wikihow,motor,Transporte ferroviario; 'motor' es una aproximación forzada.,¿Cómo abrir Las puertas de un tren?. Algunos trenes tienen botones o manijas que…
95,96,wikihow,play,Relaciones de pareja; 'play' no representa bien el tema principal.,¿Cómo saber Si el chico con el que sales es el indicado?. Debe hacerte sentir bi…


## 8. Guardado de resultados

In [16]:
if HAY_ANOTACIONES:
    resultados.round(4).to_csv(DATA / "resultados_evaluacion.csv")
    anotaciones_cat.drop(columns="text").to_csv(DATA / "anotaciones_combinadas.csv", index=False)
    print(f"Guardados resultados_evaluacion.csv y anotaciones_combinadas.csv ({TIPO_EVALUACION})")
else:
    print("Pendiente de anotación: todavía no se guardan resultados.")

Guardados resultados_evaluacion.csv y anotaciones_combinadas.csv (un anotador (anotador 2))